# 01 - Data Cleaning and Wrangling

## Project title
**Mapping the Philippine Restaurant Landscape: Food-Type Diversity, Customer Engagement, and the Distribution of Restaurant Listings Across Selected Cities**

**Objective:** Prepare the data for analysis and document the cleaning decisions.

**Scope:** Each row is a restaurant-listing record, not a complete count of all Philippine restaurants.

> Add the dataset source, collection method, and reuse terms before submission.


## Section 1 - Import the libraries

`pandas` is used to load, inspect, clean, and save the dataset.


In [61]:
import pandas as pd

## Section 2 - Load the original dataset

Load the CSV as `raw` and preview its first five rows. The original data will remain unchanged for comparison.


In [62]:
raw = pd.read_csv("Foodpanda 2025 Raw Data.csv")
raw.head()

,StoreId,CompleteStoreName,FoodType,AverageRating,Reviewers,City
0,a0ce,Marugame Udon - Lucky Chinatown Mall,Japanese,5.0,597,manila
1,a0ci,Max Mango - Bluewave Marikina,Beverages,5.0,259,marikina
2,a0cm,Flavors Lounge + Cafe - Nasipit Talamban,Coffee,0.0,0,cebu city
3,a0d1,Holy Eats! 24/7,Asian,5.0,104,pasig city
4,a0gr,McDonald's - Ormoc,Fast Food,5.0,7190,ormoc leyte


## Section 3 - Inspect the structure before cleaning

Check the dataset size, column types, missing values, and basic summaries before making changes.


In [63]:
print("Rows and columns:", raw.shape)

raw.info()
display(raw.describe(include="all"))

Rows and columns: (12409, 6)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12409 entries, 0 to 12408
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   StoreId            12409 non-null  object 
 1   CompleteStoreName  12409 non-null  object 
 2   FoodType           12318 non-null  object 
 3   AverageRating      12409 non-null  float64
 4   Reviewers          12409 non-null  int64  
 5   City               12409 non-null  object 
dtypes: float64(1), int64(1), object(4)
memory usage: 581.8+ KB


,StoreId,CompleteStoreName,FoodType,AverageRating,Reviewers,City
count,12409,12409,12318,12409.000000,12409.000000,12409
unique,12342,12406,67,NaN,NaN,18
top,fpbw,Jenlie Foods Pansiteria - Jupiter Street,Filipino,NaN,NaN,quezon city
freq,2,2,1657,NaN,NaN,2211
mean,NaN,NaN,NaN,3.715166,551.696269,NaN
std,NaN,NaN,NaN,2.124431,1448.402804,NaN
min,NaN,NaN,NaN,0.000000,0.000000,NaN
25%,NaN,NaN,NaN,3.600000,10.000000,NaN
50%,NaN,NaN,NaN,4.900000,98.000000,NaN
75%,NaN,NaN,NaN,5.000000,453.000000,NaN


## Section 4 - Diagnose actual data-quality issues

Check missing values, exact duplicate rows, and repeated store IDs. Repeated IDs are investigated but not automatically deleted because they are not exact duplicates.


In [64]:
missing_count = raw.isna().sum()
missing_percentage = raw.isna().mean() * 100

missing_summary = pd.DataFrame({
    "MissingCount": missing_count,
    "MissingPercent": missing_percentage.round(2)
})

display(missing_summary)

exact_duplicates = raw.duplicated().sum()
print("Exact duplicate rows:", exact_duplicates)

repeated_ids = raw[
    raw.duplicated(subset="StoreId", keep=False)
].sort_values("StoreId")

print("Repeated StoreId values:", repeated_ids["StoreId"].nunique())
print("Rows with repeated StoreId:", len(repeated_ids))
display(repeated_ids.head(10))

,MissingCount,MissingPercent
StoreId,0,0.00
CompleteStoreName,0,0.00
FoodType,91,0.73
AverageRating,0,0.00
Reviewers,0,0.00
City,0,0.00


Exact duplicate rows: 0
Repeated StoreId values: 67
Rows with repeated StoreId: 134


,StoreId,CompleteStoreName,FoodType,AverageRating,Reviewers,City
30,a29q,Ai - Ai Milktea Station - Country Homes Cabantian,Asian,4.9,61,davao city davao del sur
31,a29q,Ai Ai Milktea Station - Communal,Asian,4.9,57,davao city davao del sur
35,a2ds,"Pizzeria, Pasta, Burgers, Sushi & Sweets by Ch...",Sushi,4.8,518,manila
36,a2ds,Jessie's Sushi - Biak na Bato,Sushi,4.8,518,manila
51,a2uz,"Baked Sushi, Wings, Italian Pizza & Cakes The ...",Sushi,4.9,110,manila
52,a2uz,Mr. Baked Sushi - Biak na Bato,Sushi,4.9,110,manila
378,at8v,Pungko-Pungko sa Terminal,Filipino,0.0,0,cebu city
379,at8v,DC Food Hub - J Alcantara Street,Filipino,0.0,0,cebu city
395,auqj,Friends Fries - Marikina,Snacks,0.0,0,marikina
396,auqj,Friends Fries - Concepcion Uno,Snacks,0.0,0,marikina


## Section 5 - Check unrated listings

Check whether a zero rating means poor feedback or no reviews. All 3,037 zero ratings also have zero reviewers, so they are treated as unrated listings.


In [65]:
zero_rating_no_reviews = (
    (raw["AverageRating"] == 0) &
    (raw["Reviewers"] == 0)
).sum()

zero_rating_with_reviews = (
    (raw["AverageRating"] == 0) &
    (raw["Reviewers"] > 0)
).sum()

print("Zero rating and zero reviewers:", zero_rating_no_reviews)
print("Zero rating but with reviewers:", zero_rating_with_reviews)

Zero rating and zero reviewers: 3037
Zero rating but with reviewers: 0


## Section 6 - Create the cleaned working DataFrame

Create a working copy, standardize text, label missing food types as `Unknown`, and separate reviewed from unrated listings. Repeated IDs and high reviewer counts remain in the data because they may be valid.


In [66]:
clean = raw.copy()

text_columns = [
    "StoreId", "CompleteStoreName",
    "FoodType", "City"
]

for column in text_columns:
    clean[column] = clean[column].str.strip()

clean["City"] = clean["City"].str.title()
clean["FoodType"] = clean["FoodType"].fillna("Unknown")

clean["HasReviews"] = clean["Reviewers"] > 0

clean["RatingForAnalysis"] = clean["AverageRating"].where(
    clean["HasReviews"]
)

clean.head()

,StoreId,CompleteStoreName,FoodType,AverageRating,Reviewers,City,HasReviews,RatingForAnalysis
0,a0ce,Marugame Udon - Lucky Chinatown Mall,Japanese,5.0,597,Manila,True,5.0
1,a0ci,Max Mango - Bluewave Marikina,Beverages,5.0,259,Marikina,True,5.0
2,a0cm,Flavors Lounge + Cafe - Nasipit Talamban,Coffee,0.0,0,Cebu City,False,NaN
3,a0d1,Holy Eats! 24/7,Asian,5.0,104,Pasig City,True,5.0
4,a0gr,McDonald's - Ormoc,Fast Food,5.0,7190,Ormoc Leyte,True,5.0


## Section 7 - Identify reviewer-count outliers

Use the IQR method to flag unusually high reviewer counts. These values are retained because popular restaurants can genuinely receive many reviews.


In [67]:
q1 = clean["Reviewers"].quantile(0.25)
q3 = clean["Reviewers"].quantile(0.75)
iqr = q3 - q1
upper_limit = q3 + (1.5 * iqr)

reviewer_outliers = clean[
    clean["Reviewers"] > upper_limit
]

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Upper outlier limit:", upper_limit)
print("Possible reviewer outliers:", len(reviewer_outliers))
print("Maximum reviewers:", clean["Reviewers"].max())

display(
    reviewer_outliers.sort_values(
        "Reviewers", ascending=False
    ).head(10)
)

Q1: 10.0
Q3: 453.0
IQR: 443.0
Upper outlier limit: 1117.5
Possible reviewer outliers: 1517
Maximum reviewers: 30396


,StoreId,CompleteStoreName,FoodType,AverageRating,Reviewers,City,HasReviews,RatingForAnalysis
7236,q5ns,McDonald's - Concepcion Marikina,Fast Food,4.9,30396,Marikina,True,4.9
6586,p8br,Jollibee - Gaisano Grand mall mactan,Fast Food,5.0,25020,Cebu City,True,5.0
11178,x6hv,McDonald's - Malolos Bayan,Fast Food,5.0,25005,Malolos Bulacan,True,5.0
7289,q6fk,McDonald's - Davao Bajada,Fast Food,5.0,21541,Davao City Davao Del Sur,True,5.0
7909,r2mr,Ping Ping Native Lechon - Paang Bundok,Lechon,5.0,21502,Quezon City,True,5.0
7381,q7ru,Jollibee - Makati Avenue,Fast Food,5.0,20085,Makati City,True,5.0
7169,q4gm,McDonald's - JP Rizal Marikina,Fast Food,5.0,20081,Marikina,True,5.0
7000,q1on,McDonald's - Buendia,Fast Food,5.0,19021,Makati City,True,5.0
8256,r9kt,McDonald's - SPMC,Fast Food,5.0,18594,Davao City Davao Del Sur,True,5.0
7327,q6yd,McDonald's - Quirino,Fast Food,5.0,18323,Manila,True,5.0


## Section 8 - Create the cleaning log

Record each data issue, the action taken, and the reason so the cleaning decisions are clear and reproducible.


In [68]:
cleaning_log = pd.DataFrame({
    "Issue": [
        "Missing FoodType values",
        "Leading or trailing spaces",
        "Lowercase city labels",
        "Repeated StoreId values",
        "Zero ratings with zero reviewers",
        "Unusually high reviewer counts"
    ],
    "Action": [
        "Filled with Unknown",
        "Removed using str.strip()",
        "Standardized using str.title()",
        "Retained for investigation",
        "Created HasReviews and RatingForAnalysis",
        "Flagged using IQR but retained"
    ],
    "Reason": [
        "Preserves listings without inventing a food type",
        "Prevents inconsistent category labels",
        "Improves grouping and readability",
        "Repeated IDs are not exact duplicate rows",
        "Separates unrated listings from genuine ratings",
        "High review counts may be valid engagement signals"
    ]
})

display(cleaning_log)

,Issue,Action,Reason
0,Missing FoodType values,Filled with Unknown,Preserves listings without inventing a food type
1,Leading or trailing spaces,Removed using str.strip(),Prevents inconsistent category labels
2,Lowercase city labels,Standardized using str.title(),Improves grouping and readability
3,Repeated StoreId values,Retained for investigation,Repeated IDs are not exact duplicate rows
4,Zero ratings with zero reviewers,Created HasReviews and RatingForAnalysis,Separates unrated listings from genuine ratings
5,Unusually high reviewer counts,Flagged using IQR but retained,High review counts may be valid engagement sig...


## Section 9 - Validate the cleaned data

Confirm that no records were lost, missing food types were handled, ratings remain between 0 and 5, and reviewer counts are not negative. Missing values in `RatingForAnalysis` are intentional because they represent unrated listings.


In [69]:
print("Original shape:", raw.shape)
print("Cleaned shape:", clean.shape)

print("\nMissing values after cleaning:")
print(clean.isna().sum())

invalid_ratings = (
    (clean["AverageRating"] < 0) |
    (clean["AverageRating"] > 5)
).sum()

negative_reviewers = (
    clean["Reviewers"] < 0
).sum()

print("\nInvalid ratings:", invalid_ratings)
print("Negative reviewer counts:", negative_reviewers)
print("Exact duplicate rows:", clean.duplicated().sum())

Original shape: (12409, 6)
Cleaned shape: (12409, 8)

Missing values after cleaning:
StoreId                 0
CompleteStoreName       0
FoodType                0
AverageRating           0
Reviewers               0
City                    0
HasReviews              0
RatingForAnalysis    3037
dtype: int64

Invalid ratings: 0
Negative reviewer counts: 0
Exact duplicate rows: 0


## Section 10 - Save the cleaned dataset

Save the cleaned data as `Foodpanda 2025 Cleaned Data.csv`. `index=False` avoids adding an unnecessary index column.


In [70]:
clean.to_csv(
    "Foodpanda 2025 Cleaned Data.csv",
    index=False
)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.


## Presentation summary

The dataset contained 12,409 restaurant listings. We standardized text, labeled 91 missing food types as `Unknown`, and separated 3,037 unrated listings from rated listings. We retained repeated IDs and unusually high reviewer counts because they were not confirmed errors. Final validation showed that all original records remained and the cleaned data was ready for analysis.
